In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd
import numpy as np
import shap
from skl2onnx import to_onnx
import joblib
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [6]:
# Load Datasets

df = pd.read_csv("Maternal Health Risk Data Set.csv")

# Define X and y Features
X = df.drop("RiskLevel", axis=1)
y = df["RiskLevel"]

In [7]:
# Pipeline

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2


In [9]:
predictions = pipeline.predict(X_test)

In [10]:
print(f"Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")

Model Accuracy: 80.79%


In [11]:
joblib.dump(pipeline, 'maternal_care_pipeline.pkl')

['maternal_care_pipeline.pkl']

In [29]:
from sklearn.inspection import permutation_importance

# Calculate importance on the test set
# We use the pipeline so the data gets scaled correctly before checking
result = permutation_importance(
    pipeline, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

# Organize the results
perm_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance_Mean': result.importances_mean,
    'Importance_Std': result.importances_std
}).sort_values(by='Importance_Mean', ascending=False)

print(perm_importance_df)

       Feature  Importance_Mean  Importance_Std
3           BS         0.282759        0.029655
1   SystolicBP         0.143350        0.025354
0          Age         0.062069        0.011904
4     BodyTemp         0.058128        0.011823
2  DiastolicBP         0.037931        0.009865
5    HeartRate         0.023153        0.011244


In [26]:
import shap
import pandas as pd
import numpy as np

# --- PREPARATION ---
scaler   = pipeline.named_steps['scaler']
rf_model = pipeline.named_steps['classifier']

# Dynamically find the correct index for each class (no more guessing!)
class_labels = list(rf_model.classes_)          # ['high risk', 'low risk', 'mid risk']
high_risk_idx = class_labels.index('high risk') # → 0
mid_risk_idx  = class_labels.index('mid risk')  # → 2
low_risk_idx  = class_labels.index('low risk')  # → 1

print("Class order:", class_labels)
print(f"'high risk' is at index {high_risk_idx}")

# --- PICK A PATIENT ---
patient_idx    = 1
individual_data   = X_test.iloc[[patient_idx]]          # Original values (for display)
individual_scaled = scaler.transform(individual_data)   # Scaled values (for model)

# --- SHAP EXPLANATION ---
explainer   = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(individual_scaled)

# Handle both old SHAP (list) and new SHAP (single ndarray) output
if isinstance(shap_values, list):
    patient_shap = shap_values[high_risk_idx][0]
    base_value   = explainer.expected_value[high_risk_idx]
else:
    # New SHAP returns a single array for the positive class
    patient_shap = shap_values[0]
    base_value   = (explainer.expected_value
                    if np.isscalar(explainer.expected_value)
                    else explainer.expected_value[high_risk_idx])

# Predicted probabilities for all classes
proba          = rf_model.predict_proba(individual_scaled)[0]
predicted_class = class_labels[np.argmax(proba)]

# --- HUMAN-READABLE OUTPUT ---
print(f"\nPredicted class : {predicted_class}")
print(f"High Risk prob  : {proba[high_risk_idx]:.2%}")
print(f"Mid  Risk prob  : {proba[mid_risk_idx]:.2%}")
print(f"Low  Risk prob  : {proba[low_risk_idx]:.2%}")

print("\n── Top factors INCREASING high-risk score ──")
contributions = pd.DataFrame({
    'Feature': X.columns,
    'Value'  : individual_data.values[0],
    'Impact' : patient_shap
}).sort_values('Impact', ascending=False)

for _, row in contributions.iterrows():
    if row['Impact'] > 0:
        direction = "↑ raises"
        print(f"  {direction} risk  |  {row['Feature']:15s} = {row['Value']:.1f}  |  SHAP: +{row['Impact']:.3f}")

print("\n── Top factors DECREASING high-risk score ──")
for _, row in contributions[::-1].iterrows():
    if row['Impact'] < 0:
        direction = "↓ lowers"
        print(f"  {direction} risk  |  {row['Feature']:15s} = {row['Value']:.1f}  |  SHAP: {row['Impact']:.3f}")

Class order: ['high risk', 'low risk', 'mid risk']
'high risk' is at index 0

Predicted class : high risk
High Risk prob  : 100.00%
Mid  Risk prob  : 0.00%
Low  Risk prob  : 0.00%

── Top factors INCREASING high-risk score ──


ValueError: Per-column arrays must each be 1-dimensional

In [41]:
import numpy as np
import pandas as pd
from lime import lime_tabular

# 1. Your specific patient data
pheinz_physiological_data = np.array([31, 90, 60, 8, 90, 66])

# 2. Initialize the LIME Explainer
explainer = lime_tabular.LimeTabularExplainer(
    training_data=X_train.values, 
    feature_names=X.columns,
    class_names=pipeline.classes_,
    mode='classification'
)

# --- THE FIX: Create a custom prediction wrapper ---
def custom_predict_proba(data_array):
    # Convert LIME's raw array back into a DataFrame with the correct column names
    df_temp = pd.DataFrame(data_array, columns=X.columns)
    # Pass the formatted DataFrame to the pipeline
    return pipeline.predict_proba(df_temp)
# ---------------------------------------------------

# 3. Generate the Explanation using the custom wrapper
# We also explicitly tell LIME to look for the top 1 label to ensure it matches
exp = explainer.explain_instance(
    data_row=pheinz_physiological_data, 
    predict_fn=custom_predict_proba, # Use the wrapper here!
    top_labels=1
)

# Let's also print the model's actual prediction just to be 100% sure they match
actual_pred_prob = custom_predict_proba(pheinz_physiological_data.reshape(1, -1))
actual_class_idx = np.argmax(actual_pred_prob)
print(f"--- Actual Model Prediction: {pipeline.classes_[actual_class_idx]} ---")

# 4. Print the driving factors
explained_class_idx = exp.available_labels()[0]
print(f"LIME is explaining: {pipeline.classes_[explained_class_idx]}")
print("\nTop Contributing Features:")

for feature_condition, weight in exp.as_list(label=explained_class_idx):  # ← pass label here
    impact = "Increased" if weight > 0 else "Decreased"
    print(f" • {feature_condition}: {impact} probability by {abs(weight):.2%}")

--- Actual Model Prediction: mid risk ---
LIME is explaining: mid risk

Top Contributing Features:
 • BodyTemp <= 98.00: Decreased probability by 7.04%
 • SystolicBP <= 100.00: Decreased probability by 6.59%
 • 7.50 < BS <= 8.00: Decreased probability by 1.94%
 • 27.00 < Age <= 37.50: Increased probability by 1.41%
 • HeartRate <= 70.00: Decreased probability by 0.94%
 • DiastolicBP <= 65.00: Decreased probability by 0.89%


In [36]:
# Age, Systolic BP, Diastolic BP, BS, BodyTemp, HeartRate

pheinz_physiological_data = [[31, 90, 60, 15, 90, 66]]

pheinz_df = pd.DataFrame(pheinz_physiological_data, columns=X.columns)
prediction = pipeline.predict(pheinz_df)
risk_names = df["RiskLevel"].unique().tolist()


In [37]:
print(f"Pheinz Risk level is {prediction[0]}")

Pheinz Risk level is high risk


In [13]:
joblib.dump(model, 'maternal_care_model.pkl')

['maternal_care_model.pkl']

In [14]:
loaded_model = joblib.load('maternal_care_model.pkl')
predictions = loaded_model.predict(pheinz_df)

In [15]:
print(f"Pheinz risk level is {prediction[0]}")

Pheinz risk level is low risk


['maternal_care_pipeline.pkl']

In [19]:
loaded_pipe = joblib.load('maternal_care_pipeline.pkl')
predictions = loaded_model.predict(pheinz_df)
print(f"Pheinz risk level is {prediction[0]}")

Pheinz risk level is low risk


In [22]:


pipeline = joblib.load('maternal_care_pipeline.pkl')

onx = to_onnx(pipeline, X_train[:1].astype(np.float32),options={'zipmap': False})

with open("maternal_care_model.onnx", "wb") as f:
    f.write(onx.SerializeToString())

In [42]:
# Run this ONCE in your notebook after training
import json

# 50-100 rows is enough for LIME
sample = X_train.sample(100, random_state=42)
sample.to_json('training_sample.json', orient='values')

# Also save your actual class names
import json
with open('class_names.json', 'w') as f:
    json.dump(list(pipeline.classes_), f)